# ATLAS：学术任务与学习多智能体系统

## 概述
ATLAS 展示了如何构建一个面向学生学习场景的多智能体系统：用 LangGraph 把多个“专长不同的 Agent”编排成可控的工作流，为学生提供个性化的学业支持（例如日程规划、学习材料整理、学习建议）。

## 动机
学生往往需要在课程、作业、考试、社团与个人安排之间切换。传统工具常见不足：

- 难以适配个体学习风格
- 与数字日历/任务系统割裂
- 缺少上下文理解与主动建议

ATLAS 的核心思路是：把“学业支持”拆成多个职责明确的 Agent，再用结构化的 workflow 把它们协同起来。

## 关键组件
- **Coordinator Agent**：读取请求与上下文，决定需要哪些专长 Agent 参与
- **Profile Analyzer Agent**：分析学生画像（课程、偏好、约束）并提取关键要点
- **Planner Agent**：做日程/时间管理与学习计划
- **NoteWriter Agent**：把课程内容/资料整理成可复习的笔记或提纲
- **Advisor Agent**：给出个性化学习建议（节奏、方法、优先级）

## 方法概览
我们会把数据（profile/calendar/tasks）放进一个统一的 State，然后用 LangGraph 组织：

1) Coordinator 决定要调用哪些专长 Agent
2) Profile Analyzer 提取关键画像信息
3) 并行执行选中的专长 Agent（Planner/NoteWriter/Advisor）
4) 把结果写回 State，得到最终输出


## Agents 设计

![ATLAS Agent Design](../images/atlas_agent_design.png)

![ATLAS Workflow](../images/atlas_agent_workflow.png)

## 导入依赖

In [1]:
import asyncio
import json
import os
from datetime import datetime, timezone
from typing import Annotated, Any, Dict, List, Literal, Optional, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages


load_dotenv("../.env")

True

## State 定义

ATLAS 的关键在于“共享 State”：每个 node 从 state 读取输入，并把自己的输出写回 state（增量更新）。

这里我们用一个递归字典合并器，把多个 node 的结果合并到同一个 `results` 字段里。

In [2]:
def dict_reducer(dict1: Dict[str, Any], dict2: Dict[str, Any]) -> Dict[str, Any]:
    merged = dict(dict1)
    for key, value in dict2.items():
        if key in merged and isinstance(merged[key], dict) and isinstance(value, dict):
            merged[key] = dict_reducer(merged[key], value)
        else:
            merged[key] = value
    return merged


class AcademicState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    profile: Annotated[Dict[str, Any], dict_reducer]
    calendar: Annotated[Dict[str, Any], dict_reducer]
    tasks: Annotated[Dict[str, Any], dict_reducer]
    results: Annotated[Dict[str, Any], dict_reducer]

## 模型初始化

我们用统一的 DashScope 环境变量初始化 `ChatOpenAI`。

In [3]:
llm = ChatOpenAI(
    model="deepseek-v4-flash-0731",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    temperature=0,
)

## DataManager

DataManager 负责把 profile/calendar/tasks 三类结构化数据统一管理，供多个 Agent 使用。

In [4]:
class DataManager:
    def __init__(self) -> None:
        self.profile_data: dict | None = None
        self.calendar_data: dict | None = None
        self.task_data: dict | None = None

    def load_data(self, profile_json: str, calendar_json: str, task_json: str) -> None:
        self.profile_data = json.loads(profile_json)
        self.calendar_data = json.loads(calendar_json)
        self.task_data = json.loads(task_json)

    def to_state(self) -> Dict[str, Any]:
        return {
            "profile": self.profile_data or {},
            "calendar": self.calendar_data or {},
            "tasks": self.task_data or {},
        }

## Agent 节点

下面我们实现 3 类节点：

- Coordinator：决定本次请求需要哪些专长 Agent
- Profile Analyzer：把学生画像提炼成可用的“要点”
- Executor：并行运行 Planner/NoteWriter/Advisor，并把结果合并回 state


In [5]:
COORDINATOR_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a coordinator agent for an academic support system.
You must choose which specialized agents to run for the user's request.

Available agents:
- PLANNER: scheduling/time management
- NOTEWRITER: summarize/turn materials into notes
- ADVISOR: personalized academic advice

Return ONLY valid JSON with keys:
- agents: array of strings from [PLANNER, NOTEWRITER, ADVISOR]
- rationale: short string
""",
        ),
        (
            "user",
            """User request: {request}

Profile keys: {profile_keys}
Calendar keys: {calendar_keys}
Task keys: {task_keys}
""",
        ),
    ]
)


async def coordinator_node(state: AcademicState) -> Dict[str, Any]:
    request = state["messages"][-1].content if state.get("messages") else ""
    prompt = COORDINATOR_PROMPT.format_messages(
        request=request,
        profile_keys=sorted(list(state.get("profile", {}).keys())),
        calendar_keys=sorted(list(state.get("calendar", {}).keys())),
        task_keys=sorted(list(state.get("tasks", {}).keys())),
    )
    raw = (await llm.ainvoke(prompt)).content
    try:
        data = json.loads(raw)
        agents = data.get("agents", [])
        agents = [a for a in agents if a in {"PLANNER", "NOTEWRITER", "ADVISOR"}]
        if not agents:
            agents = ["PLANNER", "ADVISOR"]
        out = {"agents": agents, "rationale": data.get("rationale", "")}
    except Exception:
        out = {"agents": ["PLANNER", "ADVISOR"], "rationale": ""}

    return {"results": {"coordinator": out}}


PROFILE_ANALYZER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a profile analyzer. Summarize the student's profile into actionable bullet points.",
        ),
        (
            "user",
            """User request: {request}

Student profile (JSON):
{profile_json}
""",
        ),
    ]
)


async def profile_analyzer_node(state: AcademicState) -> Dict[str, Any]:
    request = state["messages"][-1].content if state.get("messages") else ""
    profile_json = json.dumps(state.get("profile", {}), ensure_ascii=False, indent=2)
    prompt = PROFILE_ANALYZER_PROMPT.format_messages(request=request, profile_json=profile_json)
    analysis = (await llm.ainvoke(prompt)).content
    return {"results": {"profile_analyzer": {"analysis": analysis}}}


PLANNER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a planner agent. Produce a concrete schedule/plan with time blocks and priorities.",
        ),
        (
            "user",
            """User request: {request}

Calendar (JSON):
{calendar_json}

Tasks (JSON):
{tasks_json}
""",
        ),
    ]
)


NOTEWRITER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a notewriter agent. Turn the request into a study note outline and key points.",
        ),
        ("user", "User request: {request}\n\nReturn a concise outline."),
    ]
)


ADVISOR_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an academic advisor. Give personalized advice based on the profile analysis.",
        ),
        (
            "user",
            """User request: {request}

Profile analysis:
{profile_analysis}
""",
        ),
    ]
)


async def run_planner(state: AcademicState) -> Dict[str, Any]:
    request = state["messages"][-1].content
    prompt = PLANNER_PROMPT.format_messages(
        request=request,
        calendar_json=json.dumps(state.get("calendar", {}), ensure_ascii=False, indent=2),
        tasks_json=json.dumps(state.get("tasks", {}), ensure_ascii=False, indent=2),
    )
    text = (await llm.ainvoke(prompt)).content
    return {"planner": {"plan": text}}


async def run_notewriter(state: AcademicState) -> Dict[str, Any]:
    request = state["messages"][-1].content
    prompt = NOTEWRITER_PROMPT.format_messages(request=request)
    text = (await llm.ainvoke(prompt)).content
    return {"notewriter": {"notes": text}}


async def run_advisor(state: AcademicState) -> Dict[str, Any]:
    request = state["messages"][-1].content
    profile_analysis = state.get("results", {}).get("profile_analyzer", {}).get("analysis", "")
    prompt = ADVISOR_PROMPT.format_messages(request=request, profile_analysis=profile_analysis)
    text = (await llm.ainvoke(prompt)).content
    return {"advisor": {"advice": text}}


async def executor_node(state: AcademicState) -> Dict[str, Any]:
    agents: List[str] = state.get("results", {}).get("coordinator", {}).get("agents", [])
    if not agents:
        agents = ["PLANNER", "ADVISOR"]

    tasks = []
    if "PLANNER" in agents:
        tasks.append(run_planner(state))
    if "NOTEWRITER" in agents:
        tasks.append(run_notewriter(state))
    if "ADVISOR" in agents:
        tasks.append(run_advisor(state))

    outputs = await asyncio.gather(*tasks)
    merged: Dict[str, Any] = {}
    for o in outputs:
        merged = dict_reducer(merged, o)
    return {"results": merged}

## 构建 LangGraph 工作流

我们把 3 个阶段串起来：

`coordinator → profile_analyzer → executor → END`

In [6]:
builder = StateGraph(AcademicState)
builder.add_node("coordinator", coordinator_node)
builder.add_node("profile_analyzer", profile_analyzer_node)
builder.add_node("executor", executor_node)

builder.add_edge(START, "coordinator")
builder.add_edge("coordinator", "profile_analyzer")
builder.add_edge("profile_analyzer", "executor")
builder.add_edge("executor", END)

app = builder.compile()

## 运行：给一份样例 profile/calendar/tasks + 一条请求

下面的样例数据是为了让你能直接跑通流程（你也可以替换成自己的 JSON）。

In [7]:
PROFILE_JSON = json.dumps(
    {
        "student": {"name": "Alex", "major": "Computer Science"},
        "preferences": {"learning_style": "visual", "focus_hours": ["09:00-11:30", "20:00-22:00"]},
        "academic_info": {
            "current_courses": [
                {"name": "Machine Learning", "difficulty": "high"},
                {"name": "Distributed Systems", "difficulty": "medium"},
            ]
        },
    },
    ensure_ascii=False,
)

CALENDAR_JSON = json.dumps(
    {
        "events": [
            {
                "title": "ML Lecture",
                "start": "2026-08-11T09:00:00+00:00",
                "end": "2026-08-11T10:30:00+00:00",
            },
            {
                "title": "Gym",
                "start": "2026-08-11T12:00:00+00:00",
                "end": "2026-08-11T13:00:00+00:00",
            },
        ]
    },
    ensure_ascii=False,
)

TASKS_JSON = json.dumps(
    {
        "items": [
            {"title": "ML homework 2", "due": "2026-08-13T23:59:00+00:00", "priority": "high"},
            {"title": "Read DS paper", "due": "2026-08-15T18:00:00+00:00", "priority": "medium"},
        ]
    },
    ensure_ascii=False,
)

USER_REQUEST = "I have ML homework due soon and feel distracted. Help me plan the next 3 days and give advice."

In [8]:
dm = DataManager()
dm.load_data(PROFILE_JSON, CALENDAR_JSON, TASKS_JSON)

initial_state: AcademicState = {
    "messages": [{"role": "user", "content": USER_REQUEST}],
    **dm.to_state(),
    "results": {},
}

final_state = await app.ainvoke(initial_state)
final_state["results"].keys()

dict_keys(['coordinator', 'profile_analyzer', 'planner', 'advisor'])

In [9]:
coordinator = final_state["results"].get("coordinator", {})
profile_analysis = final_state["results"].get("profile_analyzer", {}).get("analysis", "")
planner_plan = final_state["results"].get("planner", {}).get("plan", "")
notes = final_state["results"].get("notewriter", {}).get("notes", "")
advice = final_state["results"].get("advisor", {}).get("advice", "")

print("=== Coordinator ===")
print(json.dumps(coordinator, ensure_ascii=False, indent=2))
print("\n=== Profile Analyzer ===")
print(profile_analysis)
print("\n=== Planner ===")
print(planner_plan)
if notes:
    print("\n=== NoteWriter ===")
    print(notes)
print("\n=== Advisor ===")
print(advice)

=== Coordinator ===
{
  "agents": [
    "PLANNER",
    "ADVISOR"
  ],
  "rationale": "User needs scheduling for the next 3 days and personalized advice to manage distraction and ML homework."
}

=== Profile Analyzer ===
- **Prioritize Machine Learning** due to its high difficulty and upcoming deadline; allocate more study time than Distributed Systems.  
- **Schedule ML work during focus blocks**: 09:00–11:30 and 20:00–22:00, as these are your peak concentration windows.  
- **Use visual learning strategies** (e.g., diagrams, flowcharts, video explanations) to master ML concepts and reduce mental fatigue.  
- **Break ML tasks into visual chunks** (e.g., map algorithms, draw data flows) to stay engaged and minimize distraction.  
- **Reserve non-focus hours** for lighter tasks like reviewing Distributed Systems notes or quick reads.  
- **Set a 3-day plan** with clear daily goals: e.g., Day 1 – grasp core ML concepts visually; Day 2 – implement and debug code; Day 3 – finalize and submi

## 检查理解（含答案）

1) **为什么要把 profile/calendar/tasks 放进同一个 State？**
- 答：多个 Agent 都要访问同一份上下文；用 State 统一承载数据，可以让每个 node 只关注自己的输入与输出，并且能把结果稳定地合并回 state。

2) **Coordinator 在这里做了什么？**
- 答：它把“用户请求 + 上下文摘要”映射成要运行的专长 Agent 列表（例如只运行 PLANNER+ADVISOR），并把这个决策写进 `results.coordinator`。

3) **为什么 Executor 适合用并发（`asyncio.gather`）？**
- 答：Planner/NoteWriter/Advisor 彼此通常没有强依赖，可以并行调用模型，提高整体吞吐；最后再把各自结果合并回 `results`。


## 总结

- 你学习了如何用 LangGraph 为多智能体系统定义统一的 State。
- 你实现了一个 Coordinator 节点，用结构化输出决定“需要哪些专长 Agent”。
- 你实现了一个 Executor 节点，将多个专长 Agent 的调用并行执行，并把结果合并回 State。
